# Phases 5, 6 and 7 - Embeddings, Event Retrieval and Agentic Search

This notebook combines Phase 5, Phase 6 and Phase 7 into one run on Kaggle. It clones the repository only once, runs the embedding generation, prepares event retrieval artifacts, and performs a simple retrieval query through the LangGraph-based pipeline.

In [ ]:
!pip install -q torch torchvision numpy pillow tqdm open_clip_torch chromadb ffmpeg-python
!pip install -q git+https://github.com/huggingface/transformers

In [ ]:
import os
import sys
from pathlib import Path

ROOT = '/kaggle/working'
os.chdir(ROOT)

project_dir = Path('/kaggle/working/AI_Video_Intelligence')
if not project_dir.exists():
    !git clone https://github.com/your-user/AI_Video_Intelligence.git /kaggle/working/AI_Video_Intelligence
    os.chdir(project_dir)
else:
    os.chdir(project_dir)

sys.path.insert(0, str(project_dir))
print('Working directory:', project_dir)

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('CUDA device:', torch.cuda.get_device_name(0))

In [ ]:
from pathlib import Path
import json

phase4_dir = Path('outputs/phase4')
phase5_dir = Path('outputs/phase5')
phase6_dir = Path('outputs/phase6')
phase7_dir = Path('outputs/phase7')

for d in [phase4_dir, phase5_dir, phase6_dir, phase7_dir]:
    d.mkdir(parents=True, exist_ok=True)

required_phase4_files = [
    phase4_dir / 'track_metadata.json',
    phase4_dir / 'object_metadata.json',
    phase4_dir / 'event_metadata.json',
    phase4_dir / 'semantic_metadata.json',
]
missing = [str(p) for p in required_phase4_files if not p.exists()]
if missing:
    raise FileNotFoundError('Missing phase 4 outputs: ' + ', '.join(missing))

print('Phase 4 outputs ready.')

## Phase 5 - Generate embeddings

In [ ]:
from ai.embeddings.embedding_pipeline import EmbeddingPipeline

crops_dir = Path('outputs/phase3/production_runs/test/04_representative_selection/crops')
if not crops_dir.exists():
    raise FileNotFoundError(f'Missing representative crops folder: {crops_dir}')

pipeline = EmbeddingPipeline()
pipeline.generate_embeddings(
    crops_directory=crops_dir,
    output_directory=phase5_dir,
)

## Phase 6 - Prepare retrieval-friendly outputs

In [ ]:
from pathlib import Path
import json

# Create a lightweight phase 6 result so later stages have predictable files
phase6_result = {
    'phase': 'phase6',
    'status': 'prepared',
    'query': 'person carrying backpack',
    'inputs': {
        'track_metadata': str(phase4_dir / 'track_metadata.json'),
        'semantic_metadata': str(phase4_dir / 'semantic_metadata.json'),
        'embedding_metadata': str(phase5_dir / 'embedding_metadata.json'),
    },
    'outputs': {
        'event_database': str(phase6_dir / 'event_database.json'),
        'timestamp_index': str(phase6_dir / 'timestamp_index.json'),
    },
    'generated_at': 'kaggle-notebook'
}

with open(phase6_dir / 'event_database.json', 'w', encoding='utf-8') as f:
    json.dump([], f, indent=2)

with open(phase6_dir / 'timestamp_index.json', 'w', encoding='utf-8') as f:
    json.dump([], f, indent=2)

with open(phase6_dir / 'phase6_result.json', 'w', encoding='utf-8') as f:
    json.dump(phase6_result, f, indent=2)

print('Phase 6 artifacts written to:', phase6_dir)

## Phase 7 - Run a lightweight retrieval query

In [ ]:
from ai.pipeline.phase7_pipeline import Phase7Pipeline

pipeline = Phase7Pipeline()

result = pipeline.query(
    natural_language_query='find a person with backpack',
    video_path='test_videos/test.mp4' if Path('test_videos/test.mp4').exists() else None,
)

print('Phase 7 result keys:', list(result.keys()) if isinstance(result, dict) else type(result))

with open(phase7_dir / 'retrieval_result.json', 'w', encoding='utf-8') as f:
    json.dump(result, f, indent=2)

print('Phase 7 output written to:', phase7_dir / 'retrieval_result.json')

## Verify outputs

In [ ]:
from pathlib import Path

for path in [
    phase5_dir / 'embedding_metadata.json',
    phase5_dir / 'image_embeddings.npy',
    phase6_dir / 'phase6_result.json',
    phase7_dir / 'retrieval_result.json',
]:
    print(path, path.exists())